# Preprocesamiento y análisis lingüístico del texto
## Notebook 1 · Edición revisada

**Pregunta central:** ¿qué información conservamos y qué información perdemos al transformar un texto?

El preprocesamiento prepara el texto para una tarea. No existe una receta universal: eliminar un signo, cambiar mayúsculas o descartar una palabra puede ayudar en una aplicación y perjudicar en otra. Conservaremos siempre el texto original para poder comparar.

**Objetivos**
- Distinguir segmentación, normalización y anotación lingüística.
- Comparar tokenización por espacios, por reglas y mediante bibliotecas.
- Construir vocabularios y contar frecuencias con criterios explícitos.
- Interpretar lemas, categorías gramaticales, dependencias y entidades.
- Justificar las decisiones de preprocesamiento con ejemplos.

**Requisitos:** Python básico: cadenas, listas, conjuntos, diccionarios, bucles y funciones. No se requieren conocimientos de redes neuronales.

**Duración orientativa:** dos sesiones de 90 minutos, incluida la práctica. BPE es una ampliación opcional.

**Recorrido:** decisiones → tokens y vocabulario → normalización → anotación lingüística → práctica comparativa → subpalabras.

## 0. Preparación

Los primeros ejemplos utilizan únicamente la biblioteca estándar de Python. Los apartados posteriores necesitan NLTK y spaCy, junto con sus recursos lingüísticos. BPE requiere además tiktoken.

Ejecuta las siguientes celdas de instalación **solo si necesitas preparar el entorno**: descomenta sus líneas. Las descargas requieren conexión. Después, reinicia el kernel si el entorno lo solicita y ejecuta las celdas en orden. El modelo de spaCy es pequeño y funciona en CPU.

Para una clase reproducible, registra las versiones que funcionen en tu entorno junto con la versión de Python; no se presupone que cualquier combinación de versiones sea compatible.

In [ ]:
# %pip install nltk spacy tiktoken
# import sys
# !{sys.executable} -m spacy download es_core_news_sm
# import nltk
# nltk.download("punkt_tab")
# nltk.download("stopwords")

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])
for paquete in ("nltk", "spacy", "tiktoken"):
    try:
        print(f"{paquete}: {version(paquete)}")
    except PackageNotFoundError:
        print(f"{paquete}: no instalado (necesario solo en su apartado)")

## 1. Tres tipos de operaciones

| Tipo | Pregunta | Ejemplos |
|---|---|---|
| Segmentación | ¿En qué unidades dividimos el texto? | Oraciones, palabras, subpalabras |
| Normalización y filtrado | ¿Qué diferencias unificamos o descartamos? | Minúsculas, lemas, eliminación de puntuación o palabras vacías |
| Anotación lingüística | ¿Qué información añadimos? | Categorías gramaticales, dependencias, entidades |

Estas operaciones **no forman una lista de pasos obligatorios**. Por ejemplo, para reconocer entidades conviene analizar el texto conservando sus mayúsculas y contexto. Para alimentar un modelo preentrenado, debemos seguir las convenciones de su tokenizador; no aplicar automáticamente una limpieza diseñada para contar palabras.

## 2. Tokenización: ¿qué contamos como unidad?

Un **token** es una unidad producida por un tokenizador. Puede ser una palabra, un signo, una subpalabra u otra unidad. No es siempre sinónimo de palabra.

Empezamos por `split()`, que separa por espacios en blanco. Es útil para entender la idea, pero deja puntuación unida a las palabras y no separa fragmentos sin espacios.

In [ ]:
texto_original = "¿Cómo estás?Bien,gracias. Cuesta 3,50 euros."
print(texto_original.split())

### Una primera regla explícita

La siguiente expresión regular reconoce números con parte decimal, secuencias de letras y signos por separado. Es una regla didáctica, no un tokenizador universal: habrá que decidir cómo tratar correos, abreviaturas, guiones, emojis y otras escrituras.

In [ ]:
import re

PATRON = r"\d+(?:[.,]\d+)*|[^\W\d_]+|[^\w\s]"

def tokenizar_reglas(texto):
    return re.findall(PATRON, texto, flags=re.UNICODE)

print(tokenizar_reglas(texto_original))

### Ejercicio 1 · Antes de ejecutar, predice

1. Compara `split()` y `tokenizar_reglas()` con «Hola,mundo», «EE. UU.», «correo@ejemplo.es» y «teórico-práctico».
2. ¿Qué unidades te gustaría obtener en cada caso? ¿Depende de la tarea?
3. Propón una mejora de la regla y muestra un caso donde todavía falle.

**Criterio de éxito:** explicar una diferencia concreta entre los resultados, no solo imprimirlos.

## 3. Vocabulario y frecuencias

Para una tokenización y un criterio de normalización dados, el **vocabulario observado** es el conjunto de tokens distintos del corpus. No lo confundas con el vocabulario fijo de un tokenizador preentrenado.

Distinguimos **apariciones** (tokens, contando repeticiones) y **tipos** (tokens distintos). Si cambiamos las reglas, cambian ambos recuentos. Aquí conservamos únicamente tokens alfabéticos y pasamos a minúsculas.

In [ ]:
from collections import Counter

texto = "Gato, gato y perro."
tokens = tokenizar_reglas(texto)
palabras = [t.lower() for t in tokens if t.isalpha()]
vocabulario = set(palabras)
frecuencias = Counter(palabras)

print("Tokens con puntuación:", tokens)
print("Apariciones de palabras:", len(palabras))
print("Tipos de palabras:", len(vocabulario))
print("Vocabulario:", sorted(vocabulario))
print("Frecuencias:", frecuencias.most_common())

## 4. Normalizar y filtrar: decisiones con consecuencias

### 4.1. Puntuación y números

Borrar signos no equivale a separarlos. Si borramos la coma de «Hola,mundo», unimos dos palabras; si la borramos de «3,50», cambiamos la representación del número. La puntuación también puede expresar interrogación, énfasis o estructura.

In [ ]:
ejemplos = ["Hola,mundo", "Cuesta 3,50 euros", "¿Vienes?", "¡Qué bien!"]
for texto in ejemplos:
    borrado = re.sub(r"[^\w\s]", "", texto)
    print(f"Original: {texto!r}")
    print(f"  Borrar signos: {borrado!r}")
    print(f"  Separar tokens: {tokenizar_reglas(texto)}")

### 4.2. Minúsculas

`lower()` permite agrupar «Gato» y «gato», pero elimina distinciones como «ONU» y «onu» o «Lima» y «lima». Conserva el original aunque utilices una versión normalizada para búsquedas o recuentos.

In [ ]:
texto = "La ONU publicó un informe sobre Lima."
print("Original:", texto)
print("Minúsculas:", texto.lower())

### 4.3. Palabras vacías (*stop words*)

Son palabras frecuentes que podemos excluir en determinadas tareas, por ejemplo para explorar términos de un corpus. No son palabras «sin significado»: negaciones, pronombres y conectores pueden ser decisivos.

Usamos primero una lista breve para ver el algoritmo. Incluir «no» permite observar un problema, no constituye una recomendación.

In [ ]:
stop_demo = {"el", "la", "de", "este", "es", "no"}

def filtrar_demo(texto, stop_words):
    return [t.lower() for t in tokenizar_reglas(texto)
            if t.isalpha() and t.lower() not in stop_words]

for frase in ["Me gusta", "No me gusta"]:
    print(frase, "→", filtrar_demo(frase, stop_demo))

print("Conservando la negación:")
for frase in ["Me gusta", "No me gusta"]:
    print(frase, "→", filtrar_demo(frase, stop_demo - {"no"}))

El orden importa: la comparación con una lista en minúsculas debe usar una forma coherente. Conservar «no» arregla este ejemplo, pero no resuelve por sí solo toda la negación del lenguaje.

In [ ]:
tokens = ["Este", "es", "un", "ejemplo"]
print("Comparación literal:", [t for t in tokens if t not in stop_demo])
print("Comparación normalizada:", [t for t in tokens if t.lower() not in stop_demo])

### Ejercicio 2 · Elige según la tarea

Para **análisis de sentimientos**, **extracción de organizaciones** y **recuento de términos de una novela**, decide si aplicarías minúsculas, filtrado de puntuación y eliminación de palabras vacías. Justifica cada decisión con un ejemplo donde se pierda información.

No hay una única respuesta universal: importan la tarea, el modelo y la evaluación.

## 5. Tokenización y palabras vacías con NLTK

NLTK proporciona tokenizadores y recursos lingüísticos. Indicamos el español explícitamente y reutilizamos su lista de palabras vacías. Inspecciona siempre la lista antes de adoptarla.

Si aparece un `LookupError`, comprueba el nombre del recurso que solicita tu instalación y descárgalo con `nltk.download(...)`.

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

texto = "Este es un ejemplo. ¡No quiero perder la negación!"
tokens_nltk = word_tokenize(texto, language="spanish")
stop_es = set(stopwords.words("spanish"))

print("Tokens:", tokens_nltk)
print("¿La lista incluye 'no'?", "no" in stop_es)
print("Filtrado para explorar términos:",
      [t.lower() for t in tokens_nltk if t.isalpha() and t.lower() not in stop_es])

## 6. spaCy: analizar el texto en contexto

spaCy transforma un texto en un documento con tokens y anotaciones. Cargamos el modelo una sola vez y procesamos el texto original, conservando mayúsculas y signos. Sus predicciones pueden fallar: no son etiquetas infalibles.

Si falta `es_core_news_sm`, utiliza la celda de preparación. No basta con instalar la biblioteca: también necesitas el modelo.

In [ ]:
import spacy

nlp = spacy.load("es_core_news_sm")
doc = nlp("María visitó Madrid. Después compró dos libros en una librería.")
print("Oraciones:")
for oracion in doc.sents:
    print(repr(oracion.text))
print("Tokens:", [token.text for token in doc])

### 6.1. Lematización y stemming

La **lematización** busca una forma de diccionario, como «libros» → «libro». Un analizador puede utilizar información del contexto para elegirla.

El **stemming** aplica reglas para recortar formas de palabras. El resultado no tiene por qué ser una palabra válida ni coincidir con el lema. Evitamos anticipar raíces exactas: dependen del algoritmo.

Ambos métodos pueden agrupar variantes, pero también perder distinciones útiles. Comparamos sus resultados sin suponer que deban coincidir.

In [ ]:
from nltk.stem.snowball import SpanishStemmer

stemmer = SpanishStemmer()
doc_morfologia = nlp("Las mujeres corrieron mientras los corredores estaban corriendo.")
print(f"{'Token':<16} {'Lema':<16} {'Stem':<16}")
for token in doc_morfologia:
    if token.is_alpha:
        print(f"{token.text:<16} {token.lemma_:<16} {stemmer.stem(token.text):<16}")

### 6.2. Categorías gramaticales (POS)

`pos_` describe la categoría gramatical predicha: por ejemplo, sustantivo (`NOUN`), verbo (`VERB`) o nombre propio (`PROPN`). `is_alpha` indica si el token está formado por caracteres alfabéticos; no garantiza que sea una palabra de diccionario. `is_stop` señala pertenencia a la lista de palabras vacías de spaCy, que puede diferir de la de NLTK.

In [ ]:
doc_analisis = nlp("Ana compró un libro en Madrid.")
for token in doc_analisis:
    print(f"{token.text:<12} lema={token.lemma_:<12} POS={token.pos_:<6} "
          f"alfabético={token.is_alpha} stop={token.is_stop}")

### 6.3. Dependencias sintácticas

El análisis de dependencias relaciona cada token con su núcleo (`head`). `dep_` identifica la relación predicha. En lugar de memorizar una lista, inspeccionamos las etiquetas del ejemplo y pedimos su explicación.

**Pregunta:** ¿qué palabra identifica quién compra y cuál representa lo comprado? Observa el núcleo de cada relación.

In [ ]:
for token in doc_analisis:
    print(f"{token.text:<12} → {token.head.text:<12} "
          f"{token.dep_:<10} {spacy.explain(token.dep_) or ''}")

In [ ]:
from spacy import displacy

# Visualización dentro del notebook, sin iniciar un servidor.
displacy.render(doc_analisis, style="dep", jupyter=True,
                options={"compact": True, "distance": 100})

### 6.4. Entidades nombradas (NER)

NER identifica menciones como personas, lugares y organizaciones. Las etiquetas disponibles dependen del modelo. Ejecuta el ejemplo e inspecciona si las entidades y sus límites tienen sentido.

**Pregunta:** ¿qué ocurre si analizas una copia en minúsculas? El resultado puede cambiar o mantenerse; compruébalo y evita generalizar a partir de una sola frase.

In [ ]:
texto_entidades = "Ana visitó Madrid y después asistió a una reunión de la ONU."
for variante in (texto_entidades, texto_entidades.lower()):
    doc_entidades = nlp(variante)
    print("Texto:", variante)
    print([(ent.text, ent.label_) for ent in doc_entidades.ents])

displacy.render(nlp(texto_entidades), style="ent", jupyter=True)

### Ejercicio 3 · Interpretar y cuestionar

1. Analiza «El banco está junto al río» y «El banco concedió un préstamo». ¿El lema o la categoría gramatical bastan para distinguir el significado?
2. Escribe dos frases donde una misma forma tenga usos gramaticales distintos y comprueba las predicciones.
3. Busca un error o un caso ambiguo en lematización, POS o NER. Explica cuál era tu interpretación y qué produjo el modelo.

**Idea clave:** añadir anotaciones no equivale a resolver toda la comprensión del texto.

## 7. Práctica final · ¿Cómo cambian los recuentos?

Usaremos `novela.txt`, situado junto al cuaderno, para comparar cuatro representaciones:

1. Palabras en su forma original.
2. Palabras en minúsculas.
3. Palabras en minúsculas sin palabras vacías de spaCy.
4. Lemas en minúsculas, filtrando los mismos tokens vacíos que en el caso anterior.

Aquí «palabra» significa **token alfabético de spaCy**. Se excluyen números y puntuación en las cuatro variantes para que la comparación tenga un criterio común. No estamos contando todos los tokens del documento.

La lematización y las etiquetas se calculan sobre el texto original. El filtro de las variantes 3 y 4 es idéntico: así aislamos el efecto de sustituir formas por lemas.

Para empezar usamos un fragmento de hasta 20 000 caracteres, cortado al final de una línea cuando sea posible. Después puedes ampliar el análisis al archivo completo; tardará más.

In [ ]:
from pathlib import Path

ruta = Path("novela.txt")
if not ruta.exists():
    ruta = Path("Tema-02/novela.txt")
if not ruta.exists():
    raise FileNotFoundError("Sitúa novela.txt junto al notebook o ajusta la variable ruta.")

texto_novela = ruta.read_text(encoding="utf-8")
LIMITE = 20_000  # Usa None para analizar el archivo completo.
fragmento = texto_novela if LIMITE is None else texto_novela[:LIMITE]
if LIMITE is not None and len(texto_novela) > LIMITE and "\n" in fragmento:
    fragmento = fragmento.rsplit("\n", 1)[0]
print("Archivo:", ruta)
print("Caracteres analizados:", len(fragmento), "de", len(texto_novela))

In [ ]:
def construir_variantes(texto, analizador):
    variantes = {"Original": [], "Minúsculas": [],
                 "Sin stop words": [], "Lemas sin stop words": []}
    # Procesamos párrafos para evitar enviar la novela entera en una sola llamada.
    # La segmentación en párrafos limita el contexto disponible para el modelo.
    parrafos = (p for p in texto.splitlines() if p.strip())
    for documento in analizador.pipe(parrafos, batch_size=16):
        for token in documento:
            if not token.is_alpha:
                continue
            variantes["Original"].append(token.text)
            variantes["Minúsculas"].append(token.text.lower())
            if not token.is_stop:
                variantes["Sin stop words"].append(token.text.lower())
                variantes["Lemas sin stop words"].append(token.lemma_.lower())
    return variantes

variantes = construir_variantes(fragmento, nlp)

print(f"{'Representación':<24} {'Apariciones':>12} {'Tipos':>10} {'Tipos/tokens':>14}")
for nombre, palabras in variantes.items():
    total = len(palabras)
    tipos = len(set(palabras))
    proporcion = tipos / total if total else 0
    print(f"{nombre:<24} {total:>12} {tipos:>10} {proporcion:>14.3f}")

In [ ]:
def contar_frecuencias(palabras):
    return Counter(palabras)

for nombre, palabras in variantes.items():
    print("\n" + nombre)
    print(contar_frecuencias(palabras).most_common(20))

print("\nApariciones de Macondo, sin distinguir mayúsculas:",
      contar_frecuencias(variantes["Minúsculas"])["macondo"])

### Entregable y preguntas de interpretación

Entrega la tabla, las 20 formas más frecuentes de cada variante y un comentario que responda:

1. ¿Por qué pasar a minúsculas conserva el número de apariciones, pero puede reducir el vocabulario?
2. ¿Qué palabras desaparecen al filtrar? Identifica una cuya pérdida podría importar en otra tarea.
3. Localiza dos formas que se agrupen bajo un mismo lema. ¿Se produce alguna agrupación que no deseabas?
4. ¿Qué representación elegirías para explorar temas de la novela? ¿Y para reconocer sus personajes?
5. Repite con un fragmento mayor. ¿Por qué la proporción tipos/tokens no permite comparar sin más textos de longitudes distintas?

**Ampliación:** muestra las 100 formas más frecuentes, analiza el texto completo o compara los filtros de NLTK y spaCy manteniendo fija la tokenización.

**Criterios de éxito:** indicar el fragmento y las reglas utilizadas; presentar recuentos coherentes; justificar las decisiones con ejemplos del texto; reconocer al menos una limitación. Un vocabulario más pequeño no implica automáticamente un mejor modelo.

## 8. Ampliación · Subpalabras y BPE

Un vocabulario de palabras completas puede crecer mucho y dejar fuera nuevas formas. Una alternativa es representar el texto mediante unidades menores.

**Byte Pair Encoding (BPE)** aprende fusiones frecuentes de símbolos adyacentes. En una versión didáctica que parte de caracteres:

1. Dividimos las palabras en caracteres, indicando sus límites.
2. Contamos pares adyacentes, ponderados por su frecuencia en el corpus.
3. Fusionamos el par más frecuente en un símbolo nuevo.
4. Repetimos hasta alcanzar el número de fusiones elegido.

Por ejemplo, si `c a s a` aparece dos veces y `c a s o` una vez, el par `c a` aparece tres veces. Una posible primera fusión produce `ca s a` y `ca s o` (hay que fijar cómo resolver empates).

Las unidades resultantes **no tienen por qué ser morfemas ni tener significado por separado**. Existen variantes basadas en bytes y otros algoritmos de subpalabras. No todos los tokenizadores usan BPE.

### Inspeccionar un tokenizador ya entrenado

Con tiktoken no vamos a entrenar BPE: utilizamos una codificación existente. Los identificadores pertenecen a ese vocabulario concreto. Un token puede representar solo parte de los bytes de un carácter; por eso mostramos los bytes y verificamos la decodificación de la secuencia completa.

In [ ]:
import tiktoken

codificacion = tiktoken.get_encoding("cl100k_base")
texto_bpe = "¡Hola! La tokenización también procesa palabras inventadas: gatilunático."
ids = codificacion.encode(texto_bpe)

print("Tokens:", len(ids))
for token_id in ids:
    print(token_id, repr(codificacion.decode_single_token_bytes(token_id)))
print("Texto reconstruido:", codificacion.decode(ids))
assert codificacion.decode(ids) == texto_bpe

### Ejercicio opcional

Compara «casa», «casas», « casa», «Casa» y una palabra inventada. ¿Se segmentan igual? ¿Influyen los espacios o las mayúsculas? Evita deducir reglas generales a partir de un único ejemplo.

## 9. Comprobación final

Antes de dar por terminado el cuaderno, explica con tus palabras:

- Por qué un token no siempre es una palabra.
- Un caso donde eliminar puntuación cambie la información.
- Por qué una palabra frecuente puede ser esencial.
- La diferencia entre lema, stem y categoría gramatical.
- Qué reglas determinan el tamaño de un vocabulario.
- Por qué debes conservar el texto original.

**Conclusión:** una transformación se justifica por su utilidad para la tarea y por sus efectos observados, no por formar parte de una lista habitual de preprocesamiento.

### Documentación para ampliar

- [NLTK](https://www.nltk.org/)
- [Procesamiento lingüístico con spaCy](https://spacy.io/usage/linguistic-features)
- [Modelos de español de spaCy](https://spacy.io/models/es)
- [tiktoken](https://github.com/openai/tiktoken)

Estos enlaces son referencias de consulta; la compatibilidad del entorno debe comprobarse con las versiones utilizadas en clase.